# Introduction to the OpenAI Responses API

The **Responses API** is OpenAI's primary API surface. It combines the strengths of Chat Completions and the old Assistants API, and it is where the modern features live: durable **conversations**, server-side **compaction**, response **phases**, hosted tools, connectors, and WebSocket streaming.

This notebook rebuilds the introduction around the **Conversations API** as the primary multi-turn pattern, and reframes Chat Completions as legacy.

> **Model update (GPT-5.5 refresh).** All `model=` calls target **`gpt-5.5`**. `reasoning_effort` is pinned explicitly where cost/latency matters (GPT-5.5 defaults to `medium`). Light cells may use `gpt-5.5-instant` — the API name is confirmed available.

## Setup

In [1]:
import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [ ]:
!pip install --upgrade openai pandas jinja2 pydantic

## Initialize the Client

In [2]:
from openai import OpenAI

client = OpenAI()
# Make sure your OPENAI_API_KEY environment variable is set

## Basic Text Response

A single, stateless call. `instructions` sets behavior; `input` is the user turn.

In [3]:
response = client.responses.create(
    model="gpt-5.5",
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
    reasoning={"effort": "low"},
)

print(response.output_text)

Arrr! In Python, use `isinstance()`:

```python
isinstance(obj, ClassName)
```

Example:

```python
class Dog:
    pass

d = Dog()

print(isinstance(d, Dog))  # True
```

It also works with subclasses:

```python
class Animal:
    pass

class Dog(Animal):
    pass

d = Dog()

print(isinstance(d, Animal))  # True
```

To check against multiple classes, pass a tuple:

```python
isinstance(value, (int, float))
```

That returns `True` if `value` is an `int` or a `float`, matey.


## Multi-Turn with the Conversations API (the primary pattern)

Instead of manually re-sending history each turn, create a **durable conversation object** with `client.conversations.create()`. It stores messages, tool calls, and tool outputs under its own id. Pass that id on each `responses.create()` call and the model shares context automatically.

Conversations persist **indefinitely** (no 30-day TTL), unlike standalone `store: true` responses which expire after 30 days.

In [4]:
# 1) Create a durable conversation
conversation = client.conversations.create()
print("conversation id:", conversation.id)

# 2) First turn
r1 = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input="I'm designing a URL shortener. What storage would you start with?",
    reasoning={"effort": "medium"},
)
print("Turn 1:", r1.output_text[:200], "...")

# 3) Second turn -- no manual history; the conversation id carries context
r2 = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input="Now how would I add custom vanity slugs to that design?",
    reasoning={"effort": "medium"},
)
print("\nTurn 2:", r2.output_text[:200], "...")

conversation id: conv_6a281911a3e88194a4c27e2ed355c4c507697220da33061b
Turn 1: I’d start with **PostgreSQL** as the primary storage.

For a URL shortener, the core data model is simple but benefits from strong correctness:

```sql
CREATE TABLE urls (
  id BIGSERIAL PRIMARY KEY,
 ...

Turn 2: I’d treat a vanity slug as just a user-supplied `short_code`, but with stricter validation and ownership rules.

Instead of separating “generated code” and “vanity slug,” use one canonical column:

`` ...


### Legacy chaining: `previous_response_id`

Before conversations, you chained turns with `previous_response_id`. It still works for `gpt-5.5`, but prefer conversations for durable, multi-turn state.

```python
follow_up = client.responses.create(
    model="gpt-5.5",
    input="...next turn...",
    previous_response_id=r1.id,   # legacy chaining
)
```

Billing note (both patterns): prior input tokens in the chain are re-billed as input on each turn.

## Server-Side Compaction (`/responses/compact`)

Long conversations grow expensive because prior tokens are re-billed each turn. The Responses API can **compact** context server-side — shrinking what you send per turn while preserving meaning.

- **Standalone:** `client.responses.compact(model=..., input=[...])` — takes a full context window and returns a compacted version to use as input on the next call.
- **Inline:** pass `context_management=[{"type": "compaction", "compact_threshold": N}]` to `responses.create()` — the server compacts automatically when the token count crosses `N`.

In [ ]:
# Inline compaction: the server compacts context automatically when the token count crosses the threshold.
# context_management takes a list with a "compaction" entry specifying the token threshold.
r3 = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input="Summarize the whole design so far in 5 bullets.",
    reasoning={"effort": "low"},
    context_management=[{"type": "compaction", "compact_threshold": 200_000}],
)
print(r3.output_text)

## Streaming Responses

Stream a response and process `response.output_text.delta` events to accumulate text incrementally as the model generates it.

In [5]:
# Stream a response and print output text incrementally as it arrives.
stream = client.responses.create(
    model="gpt-5.5",
    input="Plan and then write a Python function to validate an email address.",
    reasoning={"effort": "medium"},
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

## Plan

1. Accept only strings.
2. Check basic size limits:
   - Full email length ≤ 254 characters.
   - Local part before `@` ≤ 64 characters.
3. Require exactly one `@`.
4. Validate the local part using a practical dot-atom pattern:
   - Allows letters, digits, and common email symbols.
   - Allows dots, but not at the beginning/end or repeated.
5. Validate the domain:
   - Supports normal domains and internationalized domains via IDNA.
   - Requires at least one dot.
   - Each label must be 1–63 characters.
   - Labels cannot start or end with `-`.
   - Rejects all-numeric top-level domains.

```python
import re


_LOCAL_PART_RE = re.compile(
    r"^[A-Za-z0-9!#$%&'*+/=?^_`{|}~-]+"
    r"(?:\.[A-Za-z0-9!#$%&'*+/=?^_`{|}~-]+)*$"
)

_DOMAIN_LABEL_RE = re.compile(
    r"^[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?$"
)


def is_valid_email(email: str) -> bool:
    """
    Validate a common email address format.

    This is practical validation, not full RFC 5322 validation.
    It 

## Transport: WebSocket Mode (low-latency streaming)

For latency-sensitive apps, the Responses API supports a **WebSocket transport**. It keeps a connection-local cache so follow-up turns (via `previous_response_id`) continue with very low latency -- ideal for voice and live agent UIs.

We don't open a socket here (it needs a running event loop and the realtime/ws client), but the pattern is: open a WS connection, send response requests over it, and read streamed events back on the same connection. See the [OpenAI Realtime API docs](https://platform.openai.com/docs/guides/realtime) for a concrete WebSocket connection pattern.

## Image Analysis

In [8]:
from IPython.display import display, Image

img_url = "https://commons.wikimedia.org/w/index.php?title=Special:FilePath&file=2023_06_08_Raccoon1.jpg&width=800"

# Visualize the image from the URL in this notebook
display(Image(url=img_url))

In [9]:
import requests
import base64



resp = requests.get(img_url, headers={"User-Agent": "Mozilla/5.0"}, allow_redirects=True)

prompt = "Extract all the visual elements of this image into a bullet points list, just output that list."

resp.raise_for_status()
img_data_url = f"data:image/jpeg;base64,{base64.b64encode(resp.content).decode()}"

response = client.responses.create(
    model="gpt-5.5",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": img_data_url},
            ],
        }
    ],
    reasoning={"effort": "low"},
)

print(response.output_text)

- Raccoon peeking from behind a tree
- Raccoon face with black mask markings
- Pointed raccoon ears
- Dark eyes and black nose
- Gray-brown fur
- Large tree trunk on the left
- Rough, textured tree bark
- Broken or cut tree stump in the foreground
- Dark forest background
- Subtle green foliage in the shadows
- High-contrast lighting with bright highlights on bark and raccoon
- Mostly black/dark negative space on the right side


## Streaming Responses

In [10]:
stream = client.responses.create(
    model="gpt-5.5",
    input="Write a one-sentence bedtime story about having pancakes as an afternoon snack.",
    reasoning={"effort": "none"},
    stream=True,
)

for event in stream:
    print(event)

ResponseCreatedEvent(response=Response(id='resp_0c2eed2a2754668d006a281ad19ac481a0b4a2a94231e53978', created_at=1781013201.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.5-2026-04-23', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, reasoning=Reasoning(effort='none', generate_summary=None, summary=None, context='current_turn'), safety_identifier=None, service_tier='auto', status='in_progress', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'), top_logprobs=0, truncation='disabled', usage=None, user=None, completed_at=None, frequency_penalty=0.0, moderation=None, presence_penalty=0.0, prompt_cache_retention='24h', store=True), sequence_number=0, type='response.created')
ResponseInProgressEvent(response=Respo

## Structured Output with Pydantic

In [11]:
from openai import OpenAI

def process_image(prompt, img_url, model="gpt-5.5"):
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": f"{img_url}"},
                ],
            }
        ],
        reasoning={"effort": "low"},
    )
    return response.output_text

In [12]:
from pydantic import BaseModel, Field

class ImageElements(BaseModel):
    subject: str = Field(description="The main subject of the image")
    general_background_description: str = Field(description="A general description of the background of the image")


def extract_structured_text(output):
    responses = client.responses.parse(
        model="gpt-5.5",
        input=[
            {"role": "system", "content": "You extract the main subject and background contained in the input."},
            {"role": "user", "content": output},
        ],
        text_format=ImageElements,
    )
    return responses.output_parsed


photo_description = "A golden-brown pancake sits on a pristine white plate, its surface glistening with syrup and a pat of melting butter. The simple background highlights the pancake, making it the clear focus of the image."
extract_structured_text(photo_description)

ImageElements(subject='A golden-brown pancake on a white plate, topped with syrup and a melting pat of butter', general_background_description='A simple, pristine white background/plate setting that keeps the focus on the pancake')

In [15]:
from IPython.display import Markdown

food_url = "https://commons.wikimedia.org/w/index.php?title=Special:FilePath&file=Pizza_01.jpg&width=800"

display(Image(url=food_url))

In [14]:
from IPython.display import Markdown

food_url = "https://commons.wikimedia.org/w/index.php?title=Special:FilePath&file=Pizza_01.jpg&width=800"
resp = requests.get(food_url, headers={"User-Agent": "Mozilla/5.0"}, allow_redirects=True)
resp.raise_for_status()

image_of_food = f"data:image/jpeg;base64,{base64.b64encode(resp.content).decode()}"

prompt = "Extract the subject and the background of this image and output that into a bullet list."

output_processed_image = process_image(prompt, image_of_food)
structured_output = extract_structured_text(output_processed_image)

Markdown(f"""
**Here's a structured summary of the image:**
- **Subject:** {structured_output.subject}
- **Background:** {structured_output.general_background_description}
""")


**Here's a structured summary of the image:**
- **Subject:** A pizza topped with assorted vegetables, including eggplant, artichokes, peppers, olives, and leafy greens.
- **Background:** A white plate on a light-colored patterned tablecloth.


## Legacy: the Chat Completions API

Chat Completions still works with `gpt-5.5`, but it is now the **legacy** surface. The conversation-state, server-side compaction, response `phase`, connectors, hosted tools, and WebSocket features above are **Responses-API-first** and are not available through Chat Completions.

Use Chat Completions only when porting old code; new work should target the Responses API.

In [ ]:
# Legacy Chat Completions (shown for migration reference only)
completion = client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"},
    ],
)
print(completion.choices[0].message.content)